# Evaluating RAG Systems

- 단순히 RAG system을 설계해서 끝났다! 로 좋은게 아니고... 시스템이 제대로 만들어진 건지 확인을 해야 함.
- 보통 **검색의 효율성부터 생성된 응답이 제대로 관련이 되있는지(관련성)까지** RAG pipeline의 다양한 측면에서의 성능, 정확성, 품질을 평가함.


그러면 왜 RAG system을 평가하는 것이 중요할까?

1. **검색 및 생성 과정의 강점과 약점을 파악**하는데 도움을 줌
2. **RAG 파이프라인 전반에 걸쳐서 어느 부분을 개선해야 하는지, 어느 부분을 최적화 해야 하는지**를 안내받을 수 있음
3. **시스템이 품질 측면에서의 기준과 사용자의 기대치를 충족하는지**를 확인할 수 있음
4. **서로 다른 RAG 구현물이나 구성 간의 비교**를 편하게 할 수 있음
5. hallucination, bias, 관련 없는 응답과 같은 **문제들을 발견**하는데 도움이 됨


Evaluation Metric에는 여러가지가 있는데, 여기선 대표적인 DeepEval을 간단하게 봐봄.

다른 소개들은 [example repo](https://github.com/adithya-s-k/AI-Engineering.academy/tree/main/docs/RAG/01_RAG_Evaluation) 참고.


In [1]:
import os
from getpass import getpass
import google.genai as genai
import nest_asyncio

nest_asyncio.apply()

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

In [9]:
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.fastembed import FastEmbedEmbedding
from llama_index.core import Settings, VectorStoreIndex, SimpleDirectoryReader

# default로 OpenAI embedding을 사용하니, Setting으로 바꿔주기.
Settings.embed_model = FastEmbedEmbedding(model_name="BAAI/bge-base-en-v1.5")
Settings.llm = GoogleGenAI(
    model = 'gemini-2.5-flash',
    api_key = GEMINI_API_KEY
)

documents = SimpleDirectoryReader("./rag_dataset").load_data()
index = VectorStoreIndex.from_documents(documents)

rag_application = index.as_query_engine()

2026-04-04 21:30:53,592 - INFO - HTTP Request: GET https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash "HTTP/1.1 200 OK"


In [12]:
# from deepeval.integrations.llama_index import FaithfulnessEvaluator

# # An example input to your RAG application
# user_input = "What is LlamaIndex?"

# # LlamaIndex returns a response object that contains
# # both the output string and retrieved nodes
# response_object = rag_application.query(user_input)

# evaluator = FaithfulnessEvaluator()
# evaluation_result = evaluator.evaluate_response(
#     query=user_input, response=response_object
# )
# print(evaluation_result)

from deepeval.metrics import FaithfulnessMetric
from deepeval.test_case import LLMTestCase
from deepeval.models.base_model import DeepEvalBaseLLM

# DeepEval도 기본적으로 OpenAI GPT 사용하니, Gemini로 바꿔주기
class DeepEvalGemini(DeepEvalBaseLLM):
    def __init__(self, model):
        self.model = model
    
    def load_model(self):
        return self.model
    
    def generate(self, prompt):
        return str(self.model.complete(prompt))

    async def a_generate(self, prompt):
        res = await self.model.acomplete(prompt)
        return str(res)

    def get_model_name(self):
        return "Gemini 2.5 Flash"

# 명시적으로 생성한 모델 생성
deepeval_gemini = DeepEvalGemini(model=Settings.llm)

# metric 초기화
metric = FaithfulnessMetric(threshold=0.7, model=deepeval_gemini)

# LlamaIndex query 실행
user_input = "What is LlamaIndex?"
response_object = rag_application.query(user_input)

# test case 생성 및 evaluation
test_case = LLMTestCase(
    input = user_input,
    actual_output = response_object.response,
    retrieval_context=[node.get_content() for node in response_object.source_nodes]
)

metric.measure(test_case)
print(f"Score: {metric.score}")
print(f"Reason: {metric.reason}")


2026-04-04 21:40:53,028 - INFO - AFC is enabled with max remote calls: 10.
2026-04-04 21:40:55,998 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


c:\Users\skdbs\.conda\envs\torch\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for Jupyter 
support
  warnings.warn('install "ipywidgets" for Jupyter support')

2026-04-04 21:40:57,093 - INFO - AFC is enabled with max remote calls: 10.
2026-04-04 21:40:57,099 - INFO - AFC is enabled with max remote calls: 10.
2026-04-04 21:41:11,971 - INFO - AFC is enabled with max remote calls: 10.
2026-04-04 21:41:14,701 - INFO - AFC is enabled with max remote calls: 10.


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 45.478928344s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-2.5-flash', 'location': 'global'},

하........

뭐 아무튼 이런식으로 deepeval 설정 바꿔주고, llama_index 설정 바꿔줘서

query를 받은 RAG system에 대한 evaluation을 진행할 수 있다고 합니다.
